In [8]:
from langchain_core.documents import Document

In [9]:

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key =os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]= os.getenv("HF_TOKEN")


In [4]:
llm =ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x12daafe00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12de24980>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name ="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12973.76it/s]


In [10]:
from langchain_chroma import Chroma
vectorstore= Chroma.from_documents(documents, embedding=embeddings)
vectorstore.similarity_search("cat")

[Document(id='eb2c15a0-58b5-4b71-b27e-e1dedf1c1fc4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3e35bedb-d923-4b72-be9b-35ddaaac8d62', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='f2d9759b-df36-42eb-abfc-d8d1af48344e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='d132cdf6-f1da-44b2-b8eb-d778a5169a13', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [11]:
#async query
await vectorstore.asimilarity_search("cats")

[Document(id='eb2c15a0-58b5-4b71-b27e-e1dedf1c1fc4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='f2d9759b-df36-42eb-abfc-d8d1af48344e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='3e35bedb-d923-4b72-be9b-35ddaaac8d62', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='d132cdf6-f1da-44b2-b8eb-d778a5169a13', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [12]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='eb2c15a0-58b5-4b71-b27e-e1dedf1c1fc4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='3e35bedb-d923-4b72-be9b-35ddaaac8d62', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='f2d9759b-df36-42eb-abfc-d8d1af48344e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956904888153076),
 (Document(id='d132cdf6-f1da-44b2-b8eb-d778a5169a13', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

In [13]:
from typing import List

In [18]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda


In [19]:
#retriever method1
retriever= RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='eb2c15a0-58b5-4b71-b27e-e1dedf1c1fc4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3e35bedb-d923-4b72-be9b-35ddaaac8d62', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [22]:
#retriever method2
retriever =vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k':1}
)
retriever.batch(["cat", "dog"])

[[Document(id='eb2c15a0-58b5-4b71-b27e-e1dedf1c1fc4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='3e35bedb-d923-4b72-be9b-35ddaaac8d62', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [23]:
#RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [24]:
message="""
Answer this question using the provided content only
{question}
context:
{context}
"""
prompt =ChatPromptTemplate.from_messages([("human"), message])

rag_chain = {"context":retriever, "question": RunnablePassthrough()}|prompt|llm
response = rag_chain.invoke("tell me about dogs")
response.content


'Dogs are great companions, known for their loyalty and friendliness.'